# 02 · Price elasticity

Phase 2 results. Every product gets two elasticities:

- **Regular price** (`ln base price`): how volume responds to a lasting price change. The optimizer uses this.
- **Promo price** (`ln(price / base)`): how volume responds to a temporary discount. Promo planning uses this.

Four models are fitted (per-product OLS, pooled OLS, MixedLM and Bayesian hierarchical). For each product, the first model in priority order that passes the quality gates (MAPE, R², bounds) is used, and failures are imputed from similar products, as in RGMx.

Logic lives in `src/elasticity/`; run `make elasticity` first.

In [ ]:
# reload edited src/ modules automatically (avoids stale imports after code changes)
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from src.config import (
    ELASTICITY_ALL_MODELS_PATH,
    ELASTICITY_PATH,
    FIGURES,
    MODEL_COMPARISON_PATH,
    Config,
)
from src.plots import ElasticityPlotter, FigureStore, PlotStyle
from src.schema import Col, Family

style = PlotStyle()
style.apply()
plotter, store = ElasticityPlotter(style), FigureStore(FIGURES)
settings = Config.load().elasticity
final = pd.read_parquet(ELASTICITY_PATH)
all_models = pd.read_parquet(ELASTICITY_ALL_MODELS_PATH)
comparison = pd.read_parquet(MODEL_COMPARISON_PATH)

## Model comparison
Holdout WAPE is measured on the last 13 weeks, never used for fitting. The share passing gates counts products whose in-sample MAPE is at most 0.3, R² at least 0.45, and regular elasticity within bounds.

In [ ]:
comparison.round(3)

In [ ]:
store.save(plotter.comparison(comparison), "02_model_comparison")

## Regular-price elasticity, product by product
Each dot is a model's estimate and each line its interval. Regular-price changes are rarer than discounts, so intervals are wider here. Partial pooling (MixedLM and Bayes) matters most for products whose own OLS is extreme.

In [ ]:
store.save(plotter.forest(all_models, final, settings.bounds), "02_elasticity_forest")

## Promo-price elasticity
The response to temporary discounts. It is usually stronger than the regular-price response, because shoppers react to a deal and also buy ahead (pantry loading).

In [ ]:
promo = plotter.forest(all_models, final, settings.promo_bounds, Family.PROMO)
store.save(promo, "02_promo_elasticity_forest")

## Final table: model selection and imputation

In [ ]:
final[Col.MODEL].fillna("imputed").value_counts()

In [ ]:
final.groupby(Col.CATEGORY)[[Col.ELASTICITY, Col.PROMO_ELASTICITY]].median().round(2)

In [ ]:
imputed = final[final[Col.ELASTICITY_SOURCE] == "imputed"]
imputed[[Col.DESCRIPTION, Col.PRODUCT_SIZE, Col.IMPUTE_REASON, Col.ELASTICITY]]